In [ ]:

"""
POWERED REPLICATION — does the Gen5 win survive?
=================================================
The V3 headline: Gen5 beat Gen1 by 9% at H=23, 3 seeds, on noisy sine.
3 seeds is a demonstration, not a finding. This runs:

  TASK 1 — noisy sine (the original task), 30 seeds
  TASK 2 — damped multi-frequency (a SECOND task it was never tuned on)

For each: Gen5 vs a parameter-MATCHED Gen1 single-substrate baseline.
Reports mean, std, paired t-test (same seed → same data → paired), and
Cohen's d effect size. The verdict rule is stated up front so it can't be
moved after seeing the numbers:

  WIN HOLDS   if Gen5 mean < Gen1 mean AND p < 0.05 AND the advantage
              is > 3% (above seed noise).
  WIN FADES   if the gap shrinks to within seed noise or p >= 0.05.

Either result is reported as-is. GPU autodetected; ~20-40 min on an H100,
longer on CPU (drop SEEDS to 10 for a quick CPU check).

Usage:  python3 powered_replication.py
        python3 powered_replication.py --seeds 10 --quick   # faster check
"""

import math, time, json, argparse, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ─────────────────────────────────────────────────────────────────────
# projections (identical to gen5_full_scope.py)
# ─────────────────────────────────────────────────────────────────────
def sphere_proj(h, r=1.0):
    nrm = h.norm(dim=-1, keepdim=True).clamp_min(1e-6)
    return torch.tanh(nrm / r) * r * (h / nrm)

def cube_proj(h, r=1.0):
    return torch.tanh(h / r) * r

def smooth_max(h, beta=8.0):
    return (1.0 / beta) * torch.logsumexp(beta * h.abs().clamp(0, 5), dim=-1, keepdim=True)

def double_well_grad(m, a, b):
    return -a * m + b * m**3

# ─────────────────────────────────────────────────────────────────────
# GEN5 — the full eight-layer substrate (faithful to gen5_full_scope.py)
# ─────────────────────────────────────────────────────────────────────
class Gen5(nn.Module):
    def __init__(self, H=23, r_outer=1.0, r_inner=0.65, R_dodec=1.2):
        super().__init__()
        self.H=H; self.r_outer=r_outer; self.r_inner=r_inner; self.R_dodec=R_dodec
        self.inp = nn.Linear(1, H)
        self.W_s  = nn.Linear(H, H, bias=False)
        self.W_px = nn.Linear(H, H, bias=False)
        self.W_py = nn.Linear(H, H, bias=False)
        self.W_pz = nn.Linear(H, H, bias=False)
        self.couple  = nn.Parameter(torch.tensor(0.30))
        self.pi_dyn  = nn.Parameter(torch.tensor(4.0))
        self.phi_dyn = nn.Parameter(torch.tensor(0.70))
        self.vesica_a= nn.Parameter(torch.tensor(0.40))
        self.vesica_b= nn.Parameter(torch.tensor(0.40))
        self.head = nn.Linear(H, 1)
    def paired_bound(self, h):
        sat = smooth_max(h)
        a = torch.sigmoid(self.pi_dyn * (sat - self.phi_dyn))
        return (1-a)*sphere_proj(h, self.r_inner) + a*cube_proj(h, self.r_outer)
    def dodec(self, h): return torch.clamp(h, -self.R_dodec, self.R_dodec)
    def forward(self, x):
        B,T,_ = x.shape; dev=x.device
        s=torch.zeros(B,self.H,device=dev); px=torch.zeros(B,self.H,device=dev)
        py=torch.zeros(B,self.H,device=dev); pz=torch.zeros(B,self.H,device=dev)
        outs=[]
        for ti in range(T):
            u=self.inp(x[:,ti])
            px_raw=self.W_px(px)+u-self.couple*(px-s) - double_well_grad(px,self.vesica_a,self.vesica_b)*0.1
            py_raw=self.W_py(py)+u-self.couple*(py-s) - double_well_grad(py,self.vesica_a,self.vesica_b)*0.1
            pz_raw=self.W_pz(pz)+u-self.couple*(pz-s) - double_well_grad(pz,self.vesica_a,self.vesica_b)*0.1
            px=self.dodec(self.paired_bound(px_raw))
            py=self.dodec(self.paired_bound(py_raw))
            pz=self.dodec(self.paired_bound(pz_raw))
            s_raw=self.W_s(s)+(px+py+pz)/3
            s=self.dodec(self.paired_bound(s_raw))
            outs.append(self.head(s))
        return torch.stack(outs,dim=1)

# ─────────────────────────────────────────────────────────────────────
# GEN1 — single bounded substrate baseline, parameter-matched
# (one recurrent matrix + paired bound + dodec cap, no p-extensions,
#  no saddle composition. the simpler architecture Gen5 must beat.)
# ─────────────────────────────────────────────────────────────────────
class Gen1(nn.Module):
    def __init__(self, H, r_outer=1.0, r_inner=0.65, R_dodec=1.2):
        super().__init__()
        self.H=H; self.r_outer=r_outer; self.r_inner=r_inner; self.R_dodec=R_dodec
        self.inp=nn.Linear(1,H)
        self.W=nn.Linear(H,H,bias=False)
        self.pi_dyn=nn.Parameter(torch.tensor(4.0))
        self.phi_dyn=nn.Parameter(torch.tensor(0.70))
        self.head=nn.Linear(H,1)
    def paired_bound(self,h):
        sat=smooth_max(h); a=torch.sigmoid(self.pi_dyn*(sat-self.phi_dyn))
        return (1-a)*sphere_proj(h,self.r_inner)+a*cube_proj(h,self.r_outer)
    def dodec(self,h): return torch.clamp(h,-self.R_dodec,self.R_dodec)
    def forward(self,x):
        B,T,_=x.shape; dev=x.device
        s=torch.zeros(B,self.H,device=dev); outs=[]
        for ti in range(T):
            u=self.inp(x[:,ti])
            s=self.dodec(self.paired_bound(self.W(s)+u))
            outs.append(self.head(s))
        return torch.stack(outs,dim=1)

def count_params(m): return sum(p.numel() for p in m.parameters())

# ─────────────────────────────────────────────────────────────────────
# TASKS
# ─────────────────────────────────────────────────────────────────────
def task_noisy_sine(B, T, noise=0.3, gen=None):
    t=torch.linspace(0,6*math.pi,T+1).unsqueeze(0).expand(B,-1)
    ph=torch.rand(B,1,generator=gen)*2*math.pi
    clean=torch.sin(t+ph)
    n=torch.randn(clean.shape,generator=gen)*noise
    return (clean+n)[:,:-1].unsqueeze(-1), clean[:,1:].unsqueeze(-1)

def task_damped_multifreq(B, T, noise=0.3, gen=None):
    """SECOND task, never tuned on: sum of two frequencies with a decaying
    envelope. Tests whether the architecture's advantage generalizes."""
    t=torch.linspace(0,6*math.pi,T+1).unsqueeze(0).expand(B,-1)
    ph1=torch.rand(B,1,generator=gen)*2*math.pi
    ph2=torch.rand(B,1,generator=gen)*2*math.pi
    env=torch.exp(-t/(6*math.pi)*1.2)
    clean=env*(0.7*torch.sin(t+ph1)+0.3*torch.sin(2.3*t+ph2))
    n=torch.randn(clean.shape,generator=gen)*noise
    return (clean+n)[:,:-1].unsqueeze(-1), clean[:,1:].unsqueeze(-1)

# ─────────────────────────────────────────────────────────────────────
# train one model, return best test loss
# ─────────────────────────────────────────────────────────────────────
def train_one(model, task_fn, seed, device, steps=1500, B=16, T=64, lr=3e-3):
    torch.manual_seed(seed)
    model=model.to(device)
    opt=torch.optim.Adam(model.parameters(), lr=lr)
    gen=torch.Generator().manual_seed(seed+10000)
    # fixed test batch per seed (same for both models → paired comparison)
    Xte,Yte=task_fn(64, T, gen=torch.Generator().manual_seed(seed+99999))
    Xte,Yte=Xte.to(device),Yte.to(device)
    best=float('inf')
    for step in range(steps):
        x,y=task_fn(B,T,gen=gen); x,y=x.to(device),y.to(device)
        loss=F.mse_loss(model(x),y)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
        opt.step()
        if step%50==0 or step==steps-1:
            with torch.no_grad():
                tl=F.mse_loss(model(Xte),Yte).item()
                if tl<best: best=tl
    return best

# ─────────────────────────────────────────────────────────────────────
# statistics
# ─────────────────────────────────────────────────────────────────────
def paired_stats(gen5_losses, gen1_losses):
    g5=np.array(gen5_losses); g1=np.array(gen1_losses)
    diff=g1-g5                      # positive = Gen5 better
    n=len(diff)
    mean_diff=diff.mean()
    sd_diff=diff.std(ddof=1)
    se=sd_diff/math.sqrt(n)
    t_stat=mean_diff/se if se>0 else float('inf')
    # two-sided p from t (normal approx for n>=30; fine here)
    from math import erf
    p=2*(1-0.5*(1+erf(abs(t_stat)/math.sqrt(2))))
    cohens_d=mean_diff/sd_diff if sd_diff>0 else float('inf')
    pct=100*mean_diff/g1.mean()
    return dict(gen5_mean=float(g5.mean()), gen5_std=float(g5.std(ddof=1)),
                gen1_mean=float(g1.mean()), gen1_std=float(g1.std(ddof=1)),
                mean_diff=float(mean_diff), pct_improvement=float(pct),
                t_stat=float(t_stat), p_value=float(p), cohens_d=float(cohens_d),
                gen5_wins=int((diff>0).sum()), n=n)

def verdict(st):
    holds = (st['gen5_mean']<st['gen1_mean']) and (st['p_value']<0.05) and (st['pct_improvement']>3.0)
    return "WIN HOLDS" if holds else "WIN FADES"

# ─────────────────────────────────────────────────────────────────────
# main
# ─────────────────────────────────────────────────────────────────────
def main():
    ap=argparse.ArgumentParser()
    ap.add_argument('--seeds',type=int,default=30)
    ap.add_argument('--steps',type=int,default=1500)
    ap.add_argument('--H',type=int,default=23)
    ap.add_argument('--quick',action='store_true',help='shorter run for a CPU sanity check')
    args,_=ap.parse_known_args()
    if args.quick: args.steps=600

    device='cuda' if torch.cuda.is_available() else 'cpu'
    if device=='cuda':
        print(f"[GPU] {torch.cuda.get_device_name(0)} — running on GPU")
    else:
        print("[CPU] No CUDA GPU detected — running on CPU (slow).")
        print("      If on a g4: install the CUDA build →")
        print("      pip install torch --index-url https://download.pytorch.org/whl/cu121")
    H=args.H; SEEDS=list(range(args.seeds))

    # param check
    g5p=count_params(Gen5(H=H));
    # match Gen1 hidden size so param counts are comparable
    g1=Gen1(H=H); g1p=count_params(g1)
    # bump Gen1 width until its params >= Gen5's (fair: simpler model gets >= budget)
    H1=H
    while count_params(Gen1(H=H1))<g5p and H1<200:
        H1+=1
    g1p=count_params(Gen1(H=H1))

    print("="*74)
    print(f"POWERED REPLICATION · device={device} · {args.seeds} seeds · {args.steps} steps")
    print(f"Gen5 H={H}: {g5p} params   |   Gen1 H={H1}: {g1p} params (matched ≥ Gen5)")
    print("VERDICT RULE (fixed before results): WIN HOLDS iff Gen5<Gen1 AND p<0.05 AND >3%")
    print("="*74)

    tasks={'noisy_sine':task_noisy_sine, 'damped_multifreq':task_damped_multifreq}
    all_results={}
    for tname,tfn in tasks.items():
        print(f"\n{'─'*74}\nTASK: {tname}\n{'─'*74}")
        g5_losses=[]; g1_losses=[]
        t0=time.time()
        for sd in SEEDS:
            l5=train_one(Gen5(H=H), tfn, sd, device, steps=args.steps)
            l1=train_one(Gen1(H=H1), tfn, sd, device, steps=args.steps)
            g5_losses.append(l5); g1_losses.append(l1)
            mark="✓" if l5<l1 else "·"
            print(f"  seed {sd:2d}  Gen5={l5:.5f}  Gen1={l1:.5f}  {mark}  "
                  f"[{time.time()-t0:.0f}s]")
        st=paired_stats(g5_losses,g1_losses)
        st['gen5_losses']=g5_losses; st['gen1_losses']=g1_losses
        v=verdict(st)
        all_results[tname]={**st,'verdict':v}
        print(f"\n  Gen5: {st['gen5_mean']:.5f} ± {st['gen5_std']:.5f}")
        print(f"  Gen1: {st['gen1_mean']:.5f} ± {st['gen1_std']:.5f}")
        print(f"  improvement: {st['pct_improvement']:+.1f}%   "
              f"Gen5 wins {st['gen5_wins']}/{st['n']} seeds")
        print(f"  paired t={st['t_stat']:.2f}  p={st['p_value']:.4g}  Cohen's d={st['cohens_d']:.2f}")
        print(f"  → {v}")

    print("\n"+"="*74); print("SUMMARY"); print("="*74)
    for tname,r in all_results.items():
        print(f"  {tname:<20} {r['pct_improvement']:+6.1f}%  p={r['p_value']:.4g}  → {r['verdict']}")
    print("\nInterpretation:")
    holds=[t for t,r in all_results.items() if r['verdict']=="WIN HOLDS"]
    if len(holds)==2:
        print("  Both tasks hold. The advantage is real and generalizes beyond the")
        print("  original task. This is the result that justifies scaling work.")
    elif 'noisy_sine' in holds:
        print("  Original task holds but second task does not. The advantage may be")
        print("  task-specific — investigate before assuming it generalizes.")
    elif holds:
        print("  Second task holds but original does not — unexpected; re-examine.")
    else:
        print("  Neither holds at this power. The 3-seed win does not survive 30 seeds.")
        print("  The substrate is a small-scale curiosity on these tasks; redirect")
        print("  effort to the ingestion perimeter, which validated independently.")

    with open('replication_results.json','w') as f:
        json.dump(all_results,f,indent=2)
    print("\nSaved replication_results.json")

if __name__=='__main__':
    main()

[GPU] NVIDIA RTX PRO 6000 Blackwell Server Edition — running on GPU
POWERED REPLICATION · device=cuda · 30 seeds · 1500 steps
Gen5 H=23: 2191 params   |   Gen1 H=46: 2257 params (matched ≥ Gen5)
VERDICT RULE (fixed before results): WIN HOLDS iff Gen5<Gen1 AND p<0.05 AND >3%

──────────────────────────────────────────────────────────────────────────
TASK: noisy_sine
──────────────────────────────────────────────────────────────────────────
  seed  0  Gen5=0.01097  Gen1=0.01077  ·  [312s]
  seed  1  Gen5=0.01182  Gen1=0.01133  ·  [592s]
  seed  2  Gen5=0.01237  Gen1=0.01182  ·  [872s]
  seed  3  Gen5=0.01158  Gen1=0.01127  ·  [1153s]
  seed  4  Gen5=0.01091  Gen1=0.01084  ·  [1433s]
  seed  5  Gen5=0.01081  Gen1=0.01127  ✓  [1712s]
  seed  6  Gen5=0.01012  Gen1=0.01036  ✓  [1992s]
  seed  7  Gen5=0.01135  Gen1=0.01140  ✓  [2272s]
  seed  8  Gen5=0.00994  Gen1=0.01015  ✓  [2552s]
  seed  9  Gen5=0.01225  Gen1=0.01228  ✓  [2831s]
  seed 10  Gen5=0.01276  Gen1=0.01280  ✓  [3110s]
  seed 11 